# Subrina — Word2Vec & Recurrent Neural Networks

**Assigned contribution:** Word2Vec representation and the recurrent neural-network family:
SimpleRNN, GRU, LSTM, Bidirectional SimpleRNN, Bidirectional GRU and Bidirectional LSTM,
including manual tuning and test evaluation.

> Shared setup/preprocessing cells are included so the notebook can run independently.
> Subrina's primary contribution begins at **Part B — Word2Vec + Recurrent Neural Networks**.


2. Dataset Upload

In [5]:
from google.colab import files

uploaded = files.upload()

!pip -q install gensim datasets transformers accelerate
!unzip -o "/content/archive (4).zip" -d "/content/"

print("Environment setup completed and dataset archive extracted.")

Saving archive (4).zip to archive (4) (1).zip
Archive:  /content/archive (4).zip
  inflating: /content/cyberbullying_tweets.csv  
Environment setup completed and dataset archive extracted.


## 3. Imports and Reproducibility



In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nltk
import tensorflow as tf
import torch

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.probability import FreqDist
from wordcloud import WordCloud

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from gensim.models import Word2Vec

from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, GRU, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

physical_gpus = tf.config.list_physical_devices('GPU')
for gpu in physical_gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

print("Imports completed.")
print("TensorFlow GPU devices:", physical_gpus)
print("PyTorch CUDA available:", torch.cuda.is_available())

Imports completed.
TensorFlow GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
PyTorch CUDA available: True


## 5. Dataset Loading and Initial Inspection

The supplied CSV contains two columns: tweet text and the cyberbullying class label.

In [7]:
DATA_PATH = "/content/cyberbullying_tweets.csv"
df = pd.read_csv(DATA_PATH)

print("Raw dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Number of classes:", df["cyberbullying_type"].nunique())
df.head()

Raw dataset shape: (47692, 2)
Columns: ['tweet_text', 'cyberbullying_type']
Number of classes: 6


,tweet_text,cyberbullying_type
0,"In other words #katandandre, your food was cra...",not_cyberbullying
1,Why is #aussietv so white? #MKR #theblock #ImA...,not_cyberbullying
2,@XochitlSuckkks a classy whore? Or more red ve...,not_cyberbullying
3,"@Jason_Gio meh. :P thanks for the heads up, b...",not_cyberbullying
4,@RudhoeEnglish This is an ISIS account pretend...,not_cyberbullying


## 6. Data Quality Analysis

The project requires missing-value and duplicate analysis. In addition to exact duplicate rows, this dataset contains repeated tweet texts. Some repeated texts are associated with more than one label, which would create contradictory supervision and possible leakage if the same text appears in different splits.

The cleanup policy is:
1. Remove rows with missing text or missing labels.
2. Identify tweet texts assigned to multiple different labels and remove all rows for those contradictory texts.
3. Remove remaining duplicate tweet texts with the same label.
4. Perform the train/validation/test split only after this cleanup.

In [8]:
missing_summary = df.isna().sum()
exact_duplicate_rows = df.duplicated().sum()
unique_texts = df["tweet_text"].nunique()
repeated_text_extra_rows = len(df) - unique_texts

label_counts_per_text = df.groupby("tweet_text")["cyberbullying_type"].nunique()
conflicting_texts = label_counts_per_text[label_counts_per_text > 1].index
conflicting_rows = df[df["tweet_text"].isin(conflicting_texts)].shape[0]

quality_summary = pd.DataFrame({
    "Measure": [
        "Raw rows",
        "Missing tweet_text",
        "Missing cyberbullying_type",
        "Exact duplicate rows",
        "Extra rows caused by repeated tweet text",
        "Tweet texts with conflicting labels",
        "Rows involved in conflicting labels"
    ],
    "Value": [
        len(df),
        missing_summary["tweet_text"],
        missing_summary["cyberbullying_type"],
        exact_duplicate_rows,
        repeated_text_extra_rows,
        len(conflicting_texts),
        conflicting_rows
    ]
})
quality_summary

,Measure,Value
0,Raw rows,47692
1,Missing tweet_text,0
2,Missing cyberbullying_type,0
3,Exact duplicate rows,36
4,Extra rows caused by repeated tweet text,1675
5,Tweet texts with conflicting labels,1639
6,Rows involved in conflicting labels,3278


In [9]:
df = df.dropna(subset=["tweet_text", "cyberbullying_type"]).copy()

label_counts_per_text = df.groupby("tweet_text")["cyberbullying_type"].nunique()
conflicting_texts = label_counts_per_text[label_counts_per_text > 1].index

df = df[~df["tweet_text"].isin(conflicting_texts)].copy()
df = df.drop_duplicates(subset=["tweet_text"]).reset_index(drop=True)

print("Dataset shape after quality cleanup:", df.shape)
print("Remaining duplicate tweet texts:", df["tweet_text"].duplicated().sum())
print("Remaining missing values:")
print(df.isna().sum())

Dataset shape after quality cleanup: (44378, 2)
Remaining duplicate tweet texts: 0
Remaining missing values:
tweet_text            0
cyberbullying_type    0
dtype: int64


## 8. Label Encoding and Stratified Train/Validation/Test Split

The cleaned dataset is split **70% / 15% / 15%**. Stratification preserves the six-class distribution in every partition and `random_state=42` makes the split reproducible.

In [10]:
label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["cyberbullying_type"])

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=df["label"]
)

validation_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=temp_df["label"]
)

train_df = train_df.reset_index(drop=True)
validation_df = validation_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Class mapping:", dict(enumerate(label_encoder.classes_)))
print("Train shape:", train_df.shape)
print("Validation shape:", validation_df.shape)
print("Test shape:", test_df.shape)

Class mapping: {0: 'age', 1: 'ethnicity', 2: 'gender', 3: 'not_cyberbullying', 4: 'other_cyberbullying', 5: 'religion'}
Train shape: (31064, 3)
Validation shape: (6657, 3)
Test shape: (6657, 3)


## 9. Preprocessing Strategies

Four preprocessing strategies are built from Lab 1 operations. Their effect is tested with the same TF-IDF + Logistic Regression validation baseline so that preprocessing is selected empirically instead of by assumption.

- **Lowercase only:** retains all symbols, hashtags and punctuation after lowercasing.
- **Alphabetic tokens:** lowercases, tokenizes and retains alphabetic tokens only.
- **Stopword + stemming:** adds English stopword removal and Porter stemming.
- **Stopword + lemmatization:** adds English stopword removal and WordNet lemmatization.

In [11]:
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
print("NLTK tokenization resources are ready.")


NLTK tokenization resources are ready.


In [12]:
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()


def preprocess_lowercase(text):
    return str(text).lower()


def preprocess_alpha(text):
    words = word_tokenize(str(text).lower())
    filtered_words = []
    for word in words:
        if word.isalpha():
            filtered_words.append(word)
    return " ".join(filtered_words)


def preprocess_stem(text):
    words = word_tokenize(str(text).lower())
    filtered_words = []
    for word in words:
        if word.isalpha() and word not in stop_words:
            filtered_words.append(stemmer.stem(word))
    return " ".join(filtered_words)


def preprocess_lemma(text):
    words = word_tokenize(str(text).lower())
    filtered_words = []
    for word in words:
        if word.isalpha() and word not in stop_words:
            filtered_words.append(lemmatizer.lemmatize(word, pos="v"))
    return " ".join(filtered_words)


preprocessing_functions = {
    "Lowercase only": preprocess_lowercase,
    "Alphabetic tokens": preprocess_alpha,
    "Stopword + stemming": preprocess_stem,
    "Stopword + lemmatization": preprocess_lemma
}

print("Preprocessing functions defined:", list(preprocessing_functions.keys()))

Preprocessing functions defined: ['Lowercase only', 'Alphabetic tokens', 'Stopword + stemming', 'Stopword + lemmatization']


### 9.1 Validation-Based Preprocessing Selection

Each preprocessing strategy is fitted only on the training split through a TF-IDF vectorizer, then evaluated on the validation split using Logistic Regression. The strategy with the highest validation **Macro-F1** is selected for the classical and recurrent-model experiments.

In [13]:
preprocessing_results = []

for name, function in preprocessing_functions.items():
    train_text = train_df["tweet_text"].apply(function)
    validation_text = validation_df["tweet_text"].apply(function)

    preprocessing_tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words="english")
    X_train_preprocessing = preprocessing_tfidf.fit_transform(train_text)
    X_validation_preprocessing = preprocessing_tfidf.transform(validation_text)

    preprocessing_model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
    preprocessing_model.fit(X_train_preprocessing, train_df["label"])
    validation_prediction = preprocessing_model.predict(X_validation_preprocessing)

    validation_accuracy = accuracy_score(validation_df["label"], validation_prediction)
    validation_f1 = f1_score(validation_df["label"], validation_prediction, average="macro")

    preprocessing_results.append({
        "Preprocessing": name,
        "Validation Accuracy": validation_accuracy,
        "Validation Macro F1": validation_f1
    })

preprocessing_results_df = pd.DataFrame(preprocessing_results).sort_values(
    "Validation Macro F1", ascending=False
).reset_index(drop=True)

preprocessing_results_df

,Preprocessing,Validation Accuracy,Validation Macro F1
0,Lowercase only,0.868259,0.855946
1,Stopword + lemmatization,0.864203,0.851797
2,Stopword + stemming,0.863302,0.850771
3,Alphabetic tokens,0.861499,0.849439


In [14]:
best_preprocessing_name = preprocessing_results_df.loc[0, "Preprocessing"]
best_preprocessing_function = preprocessing_functions[best_preprocessing_name]

train_df["clean_text"] = train_df["tweet_text"].apply(best_preprocessing_function)
validation_df["clean_text"] = validation_df["tweet_text"].apply(best_preprocessing_function)
test_df["clean_text"] = test_df["tweet_text"].apply(best_preprocessing_function)

print("Selected preprocessing strategy:", best_preprocessing_name)
print("Example cleaned tweet:")
print(train_df.loc[0, "clean_text"])

Selected preprocessing strategy: Lowercase only
Example cleaned tweet:
man this nigger dumb as fuck rt "@lumkile1st: lmao. fake weed? "@mynameztom: the cops are giving people fake weed then arresting them""


## 10. Shared Experiment Tracking and Evaluation Functions

Every required model is tuned with at least three explicit configurations. Validation Accuracy and validation Macro-F1 are recorded for every run. The final test evaluation records Accuracy, Macro-F1, the full classification report and a confusion matrix.

In [15]:
tuning_records = []
final_results = []
confusion_matrices = {}
best_hyperparameters = {}


def record_tuning(model_name, representation, config_id, parameters, validation_accuracy, validation_f1):
    tuning_records.append({
        "Model": model_name,
        "Representation": representation,
        "Config": config_id,
        "Parameters": str(parameters),
        "Validation Accuracy": validation_accuracy,
        "Validation Macro F1": validation_f1
    })


def evaluate_predictions(model_name, representation, y_true, y_pred):
    test_accuracy = accuracy_score(y_true, y_pred)
    test_f1 = f1_score(y_true, y_pred, average="macro")
    test_confusion_matrix = confusion_matrix(y_true, y_pred)

    print("=" * 90)
    print(model_name)
    print("Representation:", representation)
    print("Test Accuracy:", round(test_accuracy, 4))
    print("Test Macro F1:", round(test_f1, 4))
    print("\nFull Classification Report:\n")
    print(classification_report(y_true, y_pred, target_names=label_encoder.classes_, zero_division=0))
    print("Confusion Matrix:\n", test_confusion_matrix)

    final_results.append({
        "Model": model_name,
        "Representation": representation,
        "Accuracy": test_accuracy,
        "Macro F1": test_f1
    })
    confusion_matrices[model_name] = test_confusion_matrix

    return test_accuracy, test_f1


print("Experiment tracking and evaluation functions are ready.")

Experiment tracking and evaluation functions are ready.


In [16]:
y_train = train_df["label"].values
y_validation = validation_df["label"].values
y_test = test_df["label"].values

print("Target arrays prepared for recurrent-model training.")


Target arrays prepared for recurrent-model training.


# Part B — Word2Vec + Recurrent Neural Networks

## 15. Word2Vec Representation

Following Lab 3, a Skip-Gram Word2Vec model is trained **only on training text**. Its learned word vectors initialize the Keras embedding matrix used by all six recurrent architectures. This makes Word2Vec an evaluated classification representation rather than only a similarity demonstration.

In [17]:
train_sentences = [text.split() for text in train_df["clean_text"].astype(str).tolist()]

WORD2VEC_DIM = 50
word2vec_model = Word2Vec(vector_size=WORD2VEC_DIM, window=3, sg=1, min_count=1)
word2vec_model.build_vocab(train_sentences)
word2vec_model.train(train_sentences, total_examples=len(train_sentences), epochs=10)

print("Word2Vec vector size:", word2vec_model.vector_size)
print("Word2Vec vocabulary size:", len(word2vec_model.wv.index_to_key))

Word2Vec vector size: 50
Word2Vec vocabulary size: 79010


### 15.1 Tokenization, Padding and Word2Vec Embedding Matrix

The Lab 3 tokenizer keeps the most frequent vocabulary items and pads every tweet to a fixed sequence length. The maximum length is set to 50 tokens, matching the lab sequence-model pattern and covering the dominant region of the tweet-length distribution.

In [18]:
NUM_WORDS = 10000
MAX_LEN = 50

sequence_tokenizer = Tokenizer(num_words=NUM_WORDS, oov_token="<OOV>", filters="")
sequence_tokenizer.fit_on_texts(train_df["clean_text"].astype(str).tolist())

X_train_sequence = sequence_tokenizer.texts_to_sequences(train_df["clean_text"].astype(str).tolist())
X_validation_sequence = sequence_tokenizer.texts_to_sequences(validation_df["clean_text"].astype(str).tolist())
X_test_sequence = sequence_tokenizer.texts_to_sequences(test_df["clean_text"].astype(str).tolist())

X_train_sequence = pad_sequences(X_train_sequence, maxlen=MAX_LEN, padding="post")
X_validation_sequence = pad_sequences(X_validation_sequence, maxlen=MAX_LEN, padding="post")
X_test_sequence = pad_sequences(X_test_sequence, maxlen=MAX_LEN, padding="post")

VOCAB_SIZE = min(NUM_WORDS, len(sequence_tokenizer.word_index) + 1)
embedding_matrix = np.zeros((VOCAB_SIZE, WORD2VEC_DIM), dtype="float32")
matched_words = 0

for word, index in sequence_tokenizer.word_index.items():
    if index >= VOCAB_SIZE:
        continue
    if word in word2vec_model.wv:
        embedding_matrix[index] = word2vec_model.wv[word]
        matched_words += 1

print("Sequence train shape:", X_train_sequence.shape)
print("Vocabulary size used by Keras:", VOCAB_SIZE)
print("Word2Vec-covered vocabulary items:", matched_words)

Sequence train shape: (31064, 50)
Vocabulary size used by Keras: 10000
Word2Vec-covered vocabulary items: 9998


## 16. Recurrent Model Builder and Three Manual Configurations

All six recurrent models use the same Word2Vec embedding matrix so the architecture comparison is controlled. Three explicit configurations vary hidden units, dropout, batch size and maximum epochs. Early stopping from Lab 2 restores the best validation-loss weights within each configuration.

In [19]:
sequence_configs = [
    {"units": 32, "dropout": 0.30, "batch_size": 64, "epochs": 3},
    {"units": 64, "dropout": 0.50, "batch_size": 64, "epochs": 3},
    {"units": 64, "dropout": 0.30, "batch_size": 128, "epochs": 4}
]


def build_sequence_model(layer_type, units, dropout_rate, bidirectional=False):
    model = Sequential()
    model.add(Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=WORD2VEC_DIM,
        weights=[embedding_matrix],
        trainable=False
    ))

    if layer_type == "SimpleRNN":
        recurrent_layer = SimpleRNN(units)
    elif layer_type == "GRU":
        recurrent_layer = GRU(units)
    else:
        recurrent_layer = LSTM(units)

    if bidirectional:
        recurrent_layer = Bidirectional(recurrent_layer)

    model.add(recurrent_layer)
    model.add(Dropout(dropout_rate))
    model.add(Dense(len(label_encoder.classes_), activation="softmax"))

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model


print("Recurrent model builder is ready.")
print("Manual tuning configurations:", sequence_configs)

Recurrent model builder is ready.
Manual tuning configurations: [{'units': 32, 'dropout': 0.3, 'batch_size': 64, 'epochs': 3}, {'units': 64, 'dropout': 0.5, 'batch_size': 64, 'epochs': 3}, {'units': 64, 'dropout': 0.3, 'batch_size': 128, 'epochs': 4}]


In [20]:
def tune_and_evaluate_sequence_model(model_name, layer_type, bidirectional=False):
    best_model = None
    best_config = None
    best_validation_f1 = -1

    for config_id, config in enumerate(sequence_configs, start=1):
        tf.random.set_seed(RANDOM_STATE)
        model = build_sequence_model(
            layer_type=layer_type,
            units=config["units"],
            dropout_rate=config["dropout"],
            bidirectional=bidirectional
        )

        early_stopping = EarlyStopping(
            monitor="val_loss",
            patience=1,
            restore_best_weights=True
        )

        model.fit(
            X_train_sequence,
            y_train,
            epochs=config["epochs"],
            batch_size=config["batch_size"],
            validation_data=(X_validation_sequence, y_validation),
            callbacks=[early_stopping],
            verbose=1
        )

        validation_probabilities = model.predict(X_validation_sequence, verbose=0)
        validation_prediction = np.argmax(validation_probabilities, axis=1)
        validation_accuracy = accuracy_score(y_validation, validation_prediction)
        validation_f1 = f1_score(y_validation, validation_prediction, average="macro")

        record_tuning(model_name, "Word2Vec", config_id, config, validation_accuracy, validation_f1)
        print("Config", config_id, config, "Validation Macro F1 =", round(validation_f1, 4))

        if validation_f1 > best_validation_f1:
            best_validation_f1 = validation_f1
            best_model = model
            best_config = config

    best_hyperparameters[model_name] = best_config
    print("Best", model_name, "config:", best_config)

    test_probabilities = best_model.predict(X_test_sequence, verbose=0)
    test_prediction = np.argmax(test_probabilities, axis=1)
    evaluate_predictions(model_name, "Word2Vec", y_test, test_prediction)

    del best_model
    tf.keras.backend.clear_session()


print("Recurrent tuning/evaluation function is ready.")

Recurrent tuning/evaluation function is ready.


## 17. SimpleRNN

In [27]:
tune_and_evaluate_sequence_model("SimpleRNN", "SimpleRNN", bidirectional=False)

Epoch 1/3
486/486 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.3135 - loss: 1.5533 - val_accuracy: 0.3865 - val_loss: 1.3683
Epoch 2/3
486/486 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.4021 - loss: 1.3842 - val_accuracy: 0.5109 - val_loss: 1.2168
Epoch 3/3
486/486 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.4221 - loss: 1.3334 - val_accuracy: 0.5053 - val_loss: 1.2263
Config 1 {'units': 32, 'dropout': 0.3, 'batch_size': 64, 'epochs': 3} Validation Macro F1 = 0.46
Epoch 1/3
486/486 ━━━━━━━━━━━━━━━━━━━━ 9s 14ms/step - accuracy: 0.3811 - loss: 1.4682 - val_accuracy: 0.5214 - val_loss: 1.1978
Epoch 2/3
486/486 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.3982 - loss: 1.4767 - val_accuracy: 0.3104 - val_loss: 1.6311
Config 2 {'units': 64, 'dropout': 0.5, 'batch_size': 64, 'epochs': 3} Validation Macro F1 = 0.4663
Epoch 1/4
243/243 ━━━━━━━━━━━━━━━━━━━━ 7s 19ms/step - accuracy: 0.3876 - loss: 1.4615 - val_accuracy: 0.5169 - val_loss: 1.1107
Epoch 2/4
243/243 ━━━━━━━━━━━━━━━━━━━━ 

## 18. GRU

In [22]:
tune_and_evaluate_sequence_model("GRU", "GRU", bidirectional=False)

Epoch 1/3
486/486 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - accuracy: 0.4402 - loss: 1.2631 - val_accuracy: 0.6809 - val_loss: 0.7686
Epoch 2/3
486/486 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.7312 - loss: 0.6443 - val_accuracy: 0.7681 - val_loss: 0.5421
Epoch 3/3
486/486 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.7691 - loss: 0.5340 - val_accuracy: 0.7762 - val_loss: 0.5170
Config 1 {'units': 32, 'dropout': 0.3, 'batch_size': 64, 'epochs': 3} Validation Macro F1 = 0.7274
Epoch 1/3
486/486 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.5242 - loss: 1.0700 - val_accuracy: 0.7777 - val_loss: 0.5814
Epoch 2/3
486/486 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.7664 - loss: 0.5509 - val_accuracy: 0.7876 - val_loss: 0.5171
Epoch 3/3
486/486 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.8011 - loss: 0.4937 - val_accuracy: 0.8148 - val_loss: 0.4701
Config 2 {'units': 64, 'dropout': 0.5, 'batch_size': 64, 'epochs': 3} Validation Macro F1 = 0.7955
Epoch 1/4
243/243 ━━━━━━━━━━━━━━━━━━━━ 

## 19. LSTM

In [24]:
tune_and_evaluate_sequence_model("LSTM", "LSTM", bidirectional=False)

Epoch 1/3
486/486 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.5844 - loss: 1.0409 - val_accuracy: 0.7449 - val_loss: 0.6536
Epoch 2/3
486/486 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.7563 - loss: 0.6189 - val_accuracy: 0.7610 - val_loss: 0.5973
Epoch 3/3
486/486 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.7789 - loss: 0.5473 - val_accuracy: 0.7816 - val_loss: 0.5362
Config 1 {'units': 32, 'dropout': 0.3, 'batch_size': 64, 'epochs': 3} Validation Macro F1 = 0.7618
Epoch 1/3
486/486 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.6170 - loss: 0.9486 - val_accuracy: 0.7535 - val_loss: 0.6010
Epoch 2/3
486/486 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.7638 - loss: 0.5752 - val_accuracy: 0.7801 - val_loss: 0.5257
Epoch 3/3
486/486 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.7917 - loss: 0.5239 - val_accuracy: 0.7844 - val_loss: 0.5046
Config 2 {'units': 64, 'dropout': 0.5, 'batch_size': 64, 'epochs': 3} Validation Macro F1 = 0.7569
Epoch 1/4
243/243 ━━━━━━━━━━━━━━━━━━━━ 

## 20. Bidirectional SimpleRNN

This combines the Lab 3 `Bidirectional` wrapper pattern with the Lab 3 `SimpleRNN` layer so the required bidirectional SimpleRNN architecture is evaluated consistently with the other recurrent models.

In [25]:
tune_and_evaluate_sequence_model("Bidirectional SimpleRNN", "SimpleRNN", bidirectional=True)

Epoch 1/3
486/486 ━━━━━━━━━━━━━━━━━━━━ 12s 17ms/step - accuracy: 0.5695 - loss: 1.1070 - val_accuracy: 0.7107 - val_loss: 0.7386
Epoch 2/3
486/486 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.7128 - loss: 0.7504 - val_accuracy: 0.7283 - val_loss: 0.6712
Epoch 3/3
486/486 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.7399 - loss: 0.6783 - val_accuracy: 0.7568 - val_loss: 0.6241
Config 1 {'units': 32, 'dropout': 0.3, 'batch_size': 64, 'epochs': 3} Validation Macro F1 = 0.7409
Epoch 1/3
486/486 ━━━━━━━━━━━━━━━━━━━━ 12s 15ms/step - accuracy: 0.6088 - loss: 1.0249 - val_accuracy: 0.7265 - val_loss: 0.6848
Epoch 2/3
486/486 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.7190 - loss: 0.7336 - val_accuracy: 0.6144 - val_loss: 0.9898
Config 2 {'units': 64, 'dropout': 0.5, 'batch_size': 64, 'epochs': 3} Validation Macro F1 = 0.6945
Epoch 1/4
243/243 ━━━━━━━━━━━━━━━━━━━━ 10s 25ms/step - accuracy: 0.5902 - loss: 1.0618 - val_accuracy: 0.7158 - val_loss: 0.8099
Epoch 2/4
243/243 ━━━━━━━━━━━━━━━

## 21. Bidirectional GRU

In [26]:
tune_and_evaluate_sequence_model("Bidirectional GRU", "GRU", bidirectional=True)

Epoch 1/3
486/486 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - accuracy: 0.6557 - loss: 0.8640 - val_accuracy: 0.7922 - val_loss: 0.5091
Epoch 2/3
486/486 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.8050 - loss: 0.4848 - val_accuracy: 0.8109 - val_loss: 0.4507
Epoch 3/3
486/486 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - accuracy: 0.8208 - loss: 0.4497 - val_accuracy: 0.8262 - val_loss: 0.4268
Config 1 {'units': 32, 'dropout': 0.3, 'batch_size': 64, 'epochs': 3} Validation Macro F1 = 0.8107
Epoch 1/3
486/486 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.7099 - loss: 0.7293 - val_accuracy: 0.8055 - val_loss: 0.4779
Epoch 2/3
486/486 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - accuracy: 0.8118 - loss: 0.4760 - val_accuracy: 0.8190 - val_loss: 0.4515
Epoch 3/3
486/486 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.8229 - loss: 0.4439 - val_accuracy: 0.8217 - val_loss: 0.4370
Config 2 {'units': 64, 'dropout': 0.5, 'batch_size': 64, 'epochs': 3} Validation Macro F1 = 0.8065
Epoch 1/4
243/243 ━━━━━━━━━━━━━━━━

## 22. Bidirectional LSTM

In [28]:
tune_and_evaluate_sequence_model("Bidirectional LSTM", "LSTM", bidirectional=True)

Epoch 1/3
486/486 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.7150 - loss: 0.7548 - val_accuracy: 0.7962 - val_loss: 0.5119
Epoch 2/3
486/486 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - accuracy: 0.8028 - loss: 0.4974 - val_accuracy: 0.8140 - val_loss: 0.4595
Epoch 3/3
486/486 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.8188 - loss: 0.4561 - val_accuracy: 0.8247 - val_loss: 0.4371
Config 1 {'units': 32, 'dropout': 0.3, 'batch_size': 64, 'epochs': 3} Validation Macro F1 = 0.8093
Epoch 1/3
486/486 ━━━━━━━━━━━━━━━━━━━━ 8s 13ms/step - accuracy: 0.7392 - loss: 0.6825 - val_accuracy: 0.8025 - val_loss: 0.5047
Epoch 2/3
486/486 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.8049 - loss: 0.4967 - val_accuracy: 0.8184 - val_loss: 0.4587
Epoch 3/3
486/486 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.8177 - loss: 0.4615 - val_accuracy: 0.8185 - val_loss: 0.4527
Config 2 {'units': 64, 'dropout': 0.5, 'batch_size': 64, 'epochs': 3} Validation Macro F1 = 0.8036
Epoch 1/4
243/243 ━━━━━━━━━━━━━━━━